In [30]:
import r5py
import geopandas
import shapely
import datetime
import folium
import pandas as pd

# Recupero la path del pbf di Cagliari e del GTFS di Ctm
CA_PATH = "pbf_files/cagliari-latest.osmv2.pbf"
GTFS_PATH = "gtfs/GTFS.zip"

# Prendo due posizioni

DESTINATION = shapely.Point( 9.098612322838331, 39.23998019279815)

ORIGIN = shapely.Point(9.114625009108789, 39.22294897283518)

# Creo la transport network dei bus di Cagliari
# utilizzando il gtfs fornito da CTM Spa

ctm_network_cagliari = r5py.TransportNetwork(
    CA_PATH,
    [
        GTFS_PATH
    ]
)

# print(type(ctm_network_cagliari))


# Creo un GeoDataFrame con i punti di partenza

origins = geopandas.GeoDataFrame(
    {
        "id": [1],
        "geometry": [
            ORIGIN
        ],
    },
    crs="EPSG:4326",
)

# Creo un GeoDataFrame con i punti di destinazione

destinations = geopandas.GeoDataFrame(
    {
        "id": [2],
        "geometry": [
            DESTINATION
        ],
    },
    crs="EPSG:4326",
)

# Creo la matrice di viaggio

travel_times = r5py.TravelTimeMatrix(
    ctm_network_cagliari,
    origins=origins,
    destinations=destinations,
    departure = datetime.datetime(2025, 10, 27, 10, 22),
    transport_modes = [
        r5py.TransportMode.BUS
    ],
    snap_to_network= True,
)

detailed_itineraries = r5py.DetailedItineraries(
    ctm_network_cagliari,
    origins=origins,
    destinations=destinations,
    departure= datetime.datetime(2025, 10, 27, 10, 22),
    transport_modes=[r5py.TransportMode.BUS],
    percentiles = [1],
    snap_to_network=True,
)

print(travel_times)

print("------------------------")

print(detailed_itineraries)

print("-----------------------------------")

detailed_itineraries["mode"] = detailed_itineraries.transport_mode.astype(str)
detailed_itineraries["travel time (min)"] = detailed_itineraries.travel_time.apply(
    lambda t: round(t.total_seconds() / 60.0, 2)
)
detailed_itineraries["trip"] = detailed_itineraries.apply(
    lambda row: f"{row.from_id} → railway station",
    axis=1
)



   from_id  to_id  travel_time
0        1      2           24
------------------------
     from_id  to_id  option  segment      transport_mode      departure_time  \
0          1      2       0        0  TransportMode.WALK 2025-10-27 10:30:27   
1          1      2       0        1   TransportMode.BUS 2025-10-27 10:33:00   
2          1      2       0        2  TransportMode.WALK 2025-10-27 10:43:00   
3          1      2       1        0  TransportMode.WALK 2025-10-27 10:30:27   
4          1      2       1        1   TransportMode.BUS 2025-10-27 10:38:00   
..       ...    ...     ...      ...                 ...                 ...   
207        1      2      59        0  TransportMode.WALK 2025-10-27 10:30:27   
208        1      2      59        1   TransportMode.BUS 2025-10-27 10:38:00   
209        1      2      59        2  TransportMode.WALK 2025-10-27 10:39:31   
210        1      2      59        3   TransportMode.BUS 2025-10-27 10:45:00   
211        1      2      59      

In [35]:
single = detailed_itineraries[detailed_itineraries["option"] == 0]

detailed_routes_map = (
    single[
        [
            "geometry",
            "distance",
            "mode",
            "travel time (min)",
            "from_id",
            "to_id",
            "trip",
            "option",
            "segment",
            "route_id"
        ]
    ]
    .explore(
        tooltip=["trip", "option", "segment", "mode", "travel time (min)", "distance", "route_id"],
        column="mode",
        tiles="CartoDB.Positron",
        style_kwds={
            "weight": 3,
            "opacity": 0.8,
        },
        highlight_kwds={
            "weight": 6,
            "opacity": 1,
        },
        m= None
    )
)

# Aggiungi un marker per la stazione ferroviaria
folium.Marker(
    location=[DESTINATION.y, DESTINATION.x],
    tooltip="Destinazione",
    icon=folium.Icon(color="green", icon="cloud")  # icona funzionante
).add_to(detailed_routes_map)

# Aggiungi un marker per la stazione ferroviaria
folium.Marker(
    location=[ORIGIN.y, ORIGIN.x],
    tooltip="Partenza",
    icon=folium.Icon(color="RED", icon="cloud")  # icona funzionante
).add_to(detailed_routes_map)

# Recupero un DataFrame dal dataset di itinerari
itinerari_df = pd.DataFrame(detailed_itineraries)

segmenti_transito = itinerari_df[
    itinerari_df['transport_mode'] == r5py.TransportMode.BUS
]

C:\Users\davya\AppData\Local\Temp\ipykernel_25732\1028485534.py:45: UserWarning: color argument of Icon should be one of: {'white', 'lightgreen', 'darkred', 'lightgray', 'darkpurple', 'lightblue', 'black', 'beige', 'purple', 'lightred', 'blue', 'gray', 'green', 'cadetblue', 'red', 'pink', 'orange', 'darkgreen', 'darkblue'}.
  icon=folium.Icon(color="RED", icon="cloud")  # icona funzionante


In [36]:
detailed_routes_map

In [33]:
detailed_itineraries.head(20)

,from_id,to_id,option,segment,transport_mode,departure_time,distance,travel_time,wait_time,feed,agency_id,route_id,start_stop_id,end_stop_id,geometry,mode,travel time (min),trip
0,1,2,0,0,TransportMode.WALK,2025-10-27 10:30:27,337.531000,0 days 00:05:43,0 days 00:00:00,None,None,None,None,None,"LINESTRING (9.11482 39.22237, 9.11493 39.22238...",TransportMode.WALK,5.72,1 → railway station
1,1,2,0,1,TransportMode.BUS,2025-10-27 10:33:00,2080.172659,0 days 00:09:00,0 days 00:03:28,GTFS,500,8,BU0069,IM0103,"LINESTRING (9.11449 39.22499, 9.1104 39.22785,...",TransportMode.BUS,9.00,1 → railway station
2,1,2,0,2,TransportMode.WALK,2025-10-27 10:43:00,431.310000,0 days 00:07:18,0 days 00:00:00,None,None,None,None,None,"LINESTRING (9.10201 39.23779, 9.10201 39.23777...",TransportMode.WALK,7.30,1 → railway station
3,1,2,1,0,TransportMode.WALK,2025-10-27 10:30:27,337.531000,0 days 00:05:43,0 days 00:00:00,None,None,None,None,None,"LINESTRING (9.11482 39.22237, 9.11493 39.22238...",TransportMode.WALK,5.72,1 → railway station
4,1,2,1,1,TransportMode.BUS,2025-10-27 10:38:00,280.324799,0 days 00:00:31,0 days 00:02:02,GTFS,500,10,BU0069,BU2025,"LINESTRING (9.11449 39.22499, 9.1122 39.22678)",TransportMode.BUS,0.52,1 → railway station
5,1,2,1,2,TransportMode.WALK,2025-10-27 10:39:31,277.847000,0 days 00:04:41,0 days 00:01:04,None,None,None,None,None,"LINESTRING (9.11218 39.22674, 9.11212 39.22679...",TransportMode.WALK,4.68,1 → railway station
6,1,2,1,3,TransportMode.BUS,2025-10-27 10:45:00,188.321039,0 days 00:01:03,0 days 00:01:44,GTFS,500,5,ME0168,IM0171,"LINESTRING (9.11026 39.2277, 9.11108 39.22927)",TransportMode.BUS,1.05,1 → railway station
7,1,2,1,4,TransportMode.WALK,2025-10-27 10:47:03,1796.985000,0 days 00:30:17,0 days 00:00:00,None,None,None,None,None,"LINESTRING (9.11108 39.22927, 9.11107 39.22927...",TransportMode.WALK,30.28,1 → railway station
8,1,2,2,0,TransportMode.WALK,2025-10-27 10:30:27,337.531000,0 days 00:05:43,0 days 00:00:00,None,None,None,None,None,"LINESTRING (9.11482 39.22237, 9.11493 39.22238...",TransportMode.WALK,5.72,1 → railway station
9,1,2,2,1,TransportMode.BUS,2025-10-27 10:33:00,2080.172659,0 days 00:09:00,0 days 00:03:38,GTFS,500,8,BU0069,IM0103,"LINESTRING (9.11449 39.22499, 9.1104 39.22785,...",TransportMode.BUS,9.00,1 → railway station


In [34]:
travel_times.pivot(index="from_id", columns="to_id", values="travel_time")

to_id,2
from_id,
1,24
